# 3. Compare baseline alternatives
Goal: explore various interpolations on discrete cases (images), and compare their performance

In [4]:
#============= Globals ==========

import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.general_utils import prepare_environment, SCV_FILES, BRAIN_PLANES
from src.data_utils import load_metadata, load_volume, csv_path_to_local_path, min_max_normalize
from src.k_space_utils import image_to_kspace, kspace_to_image, zero_k_space_rows_random_dist
from src.reconstruction import interpolation_reconstruction
from src.metrices import calculate_image_quality_metrics
HPC = False

#============= Experiment Parameters  ==========

RANDOM_SEED = 1948
IMAGES_PER_PLANE = 10
RANGE_IN_PLANE_PERCENTAGE_TO_SELECT = 50
DROPPED_LINES = 70
SHOW_IMAGES = False
INTERPOLATION_TOLERANCE = 1e-3

METHODS = {
    'Zero Filling': 'zero_filling',
    'Nearest Row':  'nearest_row',
    'Linear 1D':    'linear_1d',
    'Bilinear':     'bilinear',
    'Bicubic':      'bicubic',
}
METRIC_KEYS    = ['psnr', 'ssim', 'nrmse', 'mae', 'hfen']
METRIC_DISPLAY = ['PSNR', 'SSIM', 'NRMSE', 'MAE', 'HFEN']

#============= main ==========
prepare_environment(hpc=HPC)
np.random.seed(RANDOM_SEED)

Data path: C:\mri_dataset\brain_age


## Compare the methods on a small baseline of images
Use parameters to control the experiment: number of images, range in plane percentage, dropped lines, and whether to show images.


In [5]:
# ─── helpers ────────────────────────────────────────────────────────────────

def _pick_random_slice(volume, axis, range_pct, rng_instance):
    """Return (slice_index, 2-D slice) drawn from the central range_pct % of the axis."""
    n = volume.shape[axis]
    margin = int(n * (1.0 - range_pct / 100.0) / 2)
    lo = max(0, margin)
    hi = min(n - 1, n - margin - 1)
    if lo >= hi:
        lo, hi = 0, n - 1
    idx = int(rng_instance.integers(lo, hi + 1))
    sel = [slice(None), slice(None), slice(None)]
    sel[axis] = idx
    return idx, volume[tuple(sel)]

def _safe_metrics(reference, reconstructed):
    try:
        return calculate_image_quality_metrics(reference, reconstructed)
    except Exception:
        return {k: float('nan') for k in METRIC_KEYS}

# ─── load volumes ─────────────────────────────────────────────────────────

train_df = load_metadata(SCV_FILES['train'])
print(f'Train rows: {len(train_df)}')

rng = np.random.default_rng(RANDOM_SEED)
max_trials = min(len(train_df), max(200, 20 * IMAGES_PER_PLANE))
candidate_indices = rng.permutation(len(train_df))[:max_trials]

volumes = []
for idx in candidate_indices:
    row   = train_df.iloc[idx]
    local = Path(csv_path_to_local_path(row['filePath']))
    vol   = load_volume(local)
    if vol is not None:
        volumes.append(vol)
    if len(volumes) >= IMAGES_PER_PLANE * len(BRAIN_PLANES):
        break

print(f'Loaded {len(volumes)} usable volumes')

# ─── main experiment loop ─────────────────────────────────────────────────

for plane_name, axis in BRAIN_PLANES.items():

    # accumulate per-method lists of metric dicts
    plane_metrics = {label: [] for label in METHODS}

    plane_rng  = np.random.default_rng(RANDOM_SEED + axis * 7919)
    n_cases    = min(IMAGES_PER_PLANE, len(volumes))
    vol_picks  = plane_rng.choice(len(volumes), size=n_cases, replace=False)

    for case_num, vol_idx in enumerate(vol_picks, start=1):
        volume = volumes[vol_idx]

        slice_idx, raw_slice = _pick_random_slice(
            volume, axis, RANGE_IN_PLANE_PERCENTAGE_TO_SELECT, plane_rng
        )
        normalized    = min_max_normalize(raw_slice)
        k_space       = image_to_kspace(normalized)
        harmed_k_space = zero_k_space_rows_random_dist(
            k_space, n=DROPPED_LINES,
            seed=int(plane_rng.integers(0, 2**31)),
            sigma_fraction=1/6,
        )
        damaged_image = kspace_to_image(harmed_k_space)

        # ── optional image row ────────────────────────────────────────────
        if SHOW_IMAGES:
            n_cols = 2 + len(METHODS)
            fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
            axes[0].imshow(normalized,    cmap='gray'); axes[0].set_title('Normalized')
            axes[1].imshow(damaged_image, cmap='gray'); axes[1].set_title(f'Damaged ({DROPPED_LINES} lines)')

        # ── reconstruct with every method ────────────────────────────────
        for m_idx, (method_label, method_key) in enumerate(METHODS.items()):
            recon, _ = interpolation_reconstruction(
                harmed_k_space, method=method_key, tolerance=INTERPOLATION_TOLERANCE
            )
            plane_metrics[method_label].append(_safe_metrics(normalized, recon))

            if SHOW_IMAGES:
                axes[2 + m_idx].imshow(recon, cmap='gray')
                axes[2 + m_idx].set_title(method_label)

        if SHOW_IMAGES:
            for ax in axes: ax.axis('off')
            fig.suptitle(
                f'{plane_name} | case {case_num}/{n_cases} | slice {slice_idx}',
                fontsize=12
            )
            plt.tight_layout(); plt.show()

    # ── summary table ────────────────────────────────────────────────────
    rows = {}
    for method_label, metrics_list in plane_metrics.items():
        row = {}
        for mk, md in zip(METRIC_KEYS, METRIC_DISPLAY):
            vals = np.array([m[mk] for m in metrics_list])
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                row[md] = 'N/A'
            else:
                mean = vals.mean()
                std  = vals.std(ddof=0)
                row[md] = f'{mean:.1f}±{std:.1f}'
        rows[method_label] = row

    summary_df = pd.DataFrame(rows).T[METRIC_DISPLAY]
    print(f'\n=== {plane_name} plane  |  {n_cases} images  |  {DROPPED_LINES} dropped lines ===')
    display(summary_df)


Train rows: 4233
Loaded 30 usable volumes

=== Sagittal plane  |  10 images  |  70 dropped lines ===


,PSNR,SSIM,NRMSE,MAE,HFEN
Zero Filling,25.6±1.8,0.7±0.0,0.2±0.0,0.0±0.0,0.4±0.0
Nearest Row,24.0±1.7,0.6±0.0,0.3±0.0,0.0±0.0,0.5±0.1
Linear 1D,24.9±1.7,0.6±0.0,0.2±0.0,0.0±0.0,0.4±0.1
Bilinear,24.9±1.7,0.6±0.0,0.2±0.0,0.0±0.0,0.4±0.1
Bicubic,24.1±1.8,0.6±0.0,0.4±0.1,0.1±0.0,0.5±0.1



=== Coronal plane  |  10 images  |  70 dropped lines ===


,PSNR,SSIM,NRMSE,MAE,HFEN
Zero Filling,20.7±0.7,0.7±0.0,0.2±0.0,0.0±0.0,0.6±0.1
Nearest Row,18.0±2.8,0.6±0.1,0.5±0.2,0.1±0.0,0.8±0.1
Linear 1D,19.2±1.6,0.6±0.0,0.4±0.1,0.1±0.0,0.8±0.1
Bilinear,19.2±1.6,0.6±0.0,0.4±0.1,0.1±0.0,0.8±0.1
Bicubic,17.2±2.9,0.5±0.1,0.6±0.2,0.1±0.0,1.0±0.4



=== Axial plane  |  10 images  |  70 dropped lines ===


,PSNR,SSIM,NRMSE,MAE,HFEN
Zero Filling,21.1±1.7,0.7±0.0,0.3±0.1,0.1±0.0,0.6±0.0
Nearest Row,19.5±2.2,0.6±0.0,0.4±0.1,0.1±0.0,0.8±0.1
Linear 1D,20.3±2.0,0.6±0.0,0.4±0.1,0.1±0.0,0.7±0.1
Bilinear,20.3±2.0,0.6±0.0,0.4±0.1,0.1±0.0,0.7±0.1
Bicubic,19.2±2.5,0.5±0.1,0.5±0.2,0.1±0.1,1.0±0.5
